# ACL Python 算子调用：GEMM

## 5.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
command -v atc
npu-smi info


<a id="s-05-02_gemm"></a>


## 5.2 GEMM：`acl.blas.gemm_ex`

### 小节概述

目标：先用 ATC 编译 GEMM 单算子 OM，再通过 `acl.blas.gemm_ex` 提交 GEMM，读回结果并与 NumPy reference 比较。

<table style="text-align:left; margin-left:0;">
<tr><th>参数/对象</th><th>本实验含义</th></tr>
<tr><td><code>M/N/K</code></td><td>输出行数、输出列数、归约维</td></tr>
<tr><td><code>float16</code></td><td>A、B、C、alpha、beta 的存储类型</td></tr>
<tr><td><code>ACL_COMPUTE_HIGH_PRECISION</code></td><td><code>gemm_ex</code> 的计算精度策略</td></tr>
<tr><td>Stream</td><td>异步提交算子；必须同步后才能 D2H</td></tr>
<tr><td>Event</td><td>记录同一 Stream 上重复算子的设备时间，不包含 ATC、H2D、D2H</td></tr>
<tr><td>模型目录</td><td>ATC 单算子 OM 所在目录，由 <code>acl.op.set_model_dir</code> 注册</td></tr>
</table>


### 与上一章的接口区别

| 接口 | 本课程的准备方式 | 输入格式 |
|---|---|---|
| C++ ACLNN GEMM | 查询 workspace，准备 executor | 稠密 Tensor |
| C++ ops-sparse SpMV | 创建 handle 和描述符 | CSR |
| Python gemm_ex / execute_v2 | 用 ATC 生成单算子 OM，注册模型目录 | 稠密矩阵 / COO |

本节的 OM 是这条 Python 接口路径需要的准备文件，并非所有 Python 算子接口都采用同一种方式。


In [ ]:
import json
import subprocess
import time
from pathlib import Path

import acl
import numpy as np

%run src/acl_helpers.py

SOC_VERSION = acl.get_soc_name()

# 学生修改：矩阵大小、设备编号和重复次数。
M, N, K = 64, 64, 64
DEVICE_ID = 0
REPEAT = 10
MODEL_DIR = (Path("work") / "05.02_gemm" / "models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"M={M}, N={N}, K={K}, dtype=float16, device={DEVICE_ID}, repeat={REPEAT}, soc={SOC_VERSION}")


### 准备输入

修改并运行上面的参数单元格后，从这里向下重新运行。输入形状和单算子描述会使用同一组参数。


In [ ]:
a_host = ((np.arange(M)[:, None] * 3 + np.arange(K)[None, :]) % 7).astype(np.float16)
b_host = ((np.arange(K)[:, None] * 2 + np.arange(N)[None, :]) % 9).astype(np.float16)
c_host = np.zeros((M, N), dtype=np.float16)
alpha_host = np.array([1.0], dtype=np.float16)
beta_host = np.array([0.0], dtype=np.float16)


### 生成并编译单算子描述

ATC 把算子描述编译为适配当前 SoC（`SOC_VERSION`）的 OM。后续 `acl.op.set_model_dir` 只注册该目录，不加载整图模型。


In [ ]:
gemm_singleop = [{
    "op": "GEMM",
    "input_desc": [
        {"format": "ND", "shape": [M, K], "type": "float16"},  # a
        {"format": "ND", "shape": [K, N], "type": "float16"},  # b
        {"format": "ND", "shape": [M, N], "type": "float16"},  # c（输出就地写入）
        {"format": "ND", "shape": [], "type": "float16"},      # alpha：标量
        {"format": "ND", "shape": [], "type": "float16"},      # beta：标量
    ],
    "output_desc": [{"format": "ND", "shape": [M, N], "type": "float16"}],
    "attr": [
        {"name": "transpose_a", "type": "bool", "value": False},
        {"name": "transpose_b", "type": "bool", "value": False},
    ],
}]

json_path = MODEL_DIR / "gemm_singleop.json"
json_path.write_text(json.dumps(gemm_singleop, indent=2, ensure_ascii=False), encoding="utf-8")
print("已写入", json_path)


In [ ]:
subprocess.run([
    "atc", f"--singleop={json_path}", f"--output={MODEL_DIR}",
    f"--soc_version={SOC_VERSION}",
], check=True)


### 在设备上计算

`AclSession` 创建 Context、Stream 和 Event，并在这段代码结束或报错时释放资源。
`to_device` 负责申请设备内存并拷入数据。
这些重复操作的完整代码见 [acl_helpers.py](src/acl_helpers.py)。

下面保留算子调用、Event 计时、同步和结果回读。阅读时重点看传入算子的参数，以及同步发生的位置。

输入由 `acl_helpers.py` 将数组转成 bytes，再用 `bytes_to_ptr` 完成同步 H2D。输出先复制到 `malloc_host` 申请的 Host 内存，再用 `ptr_to_bytes` 和 `np.frombuffer` 还原数组。


`gemm_ex` 的位置参数按下面几组阅读：

| 参数组 | 本例设置 |
|---|---|
| trans_a、trans_b、trans_c | 0：不转置 |
| M、N、K | A[M,K] × B[K,N] → C[M,N] |
| alpha、A、B、beta、C | 设备地址；C 在原缓冲区更新 |
| lda、ldb、ldc | 本接口的保留参数，设为 -1 |
| data_type、compute、stream | FP16 存储、高精度计算策略、当前 Stream |

公式为 `C = alpha × A × B + beta × C`。与上一章不同，这里直接更新 C。参数定义见[gemm_ex 接口说明](https://www.hiascend.com/doc_center/source/en/canncommercial/800/apiref/appdevgapi/aclpythondevg_01_0818.html)。


In [ ]:
with AclSession(MODEL_DIR, DEVICE_ID) as session:
    dev_a = session.to_device(a_host)
    dev_b = session.to_device(b_host)
    dev_c = session.to_device(c_host)
    dev_alpha = session.to_device(alpha_host)
    dev_beta = session.to_device(beta_host)

    # 重复算子调用，Event 只统计这一段设备执行时间。
    check(acl.rt.record_event(session.start_event, session.stream), "acl.rt.record_event start")
    for _ in range(REPEAT):
        check(acl.blas.gemm_ex(
            0, 0, 0,
            M, N, K,
            dev_alpha, dev_a, -1, ACL_FLOAT16,
            dev_b, -1, ACL_FLOAT16,
            dev_beta, dev_c, -1, ACL_FLOAT16,
            ACL_COMPUTE_HIGH_PRECISION, session.stream,
        ), "acl.blas.gemm_ex")
    check(acl.rt.record_event(session.end_event, session.stream), "acl.rt.record_event end")
    check(acl.rt.synchronize_stream(session.stream), "acl.rt.synchronize_stream")
    elapsed_ms, ret = acl.rt.event_elapsed_time(session.start_event, session.end_event)
    check(ret, "acl.rt.event_elapsed_time")

    out_host = np.empty((M, N), dtype=np.float16)
    host_ptr, ret = acl.rt.malloc_host(out_host.nbytes)
    check(ret, "acl.rt.malloc_host")
    try:
        check(acl.rt.memcpy(
            host_ptr, out_host.nbytes,
            dev_c, c_host.nbytes, MEMCPY_D2H,
        ), "acl.rt.memcpy D2H")
        result_bytes = acl.util.ptr_to_bytes(host_ptr, out_host.nbytes)
    finally:
        acl.rt.free_host(host_ptr)
    out_host = np.frombuffer(result_bytes, dtype=out_host.dtype).reshape(out_host.shape)


### 与 NumPy 结果比较

最大绝对误差取所有元素中最大的 `|设备结果−参考结果|`；前一章的相对 L2 则用整体差值的长度除以参考结果长度，两种阈值不能直接比较。本例输入和输出为 FP16，使用 `atol=0.02, rtol=0`。

Event 包围设备重复调用，参考时间则来自一次 NumPy 计算。这里先看校验是否通过；两边没有相同的预热和重复方式，不能直接据此得出稳定加速比。


In [ ]:
reference_start = time.perf_counter()
golden = (a_host.astype(np.float32) @ b_host.astype(np.float32)).astype(np.float16)
reference_ms = (time.perf_counter() - reference_start) * 1000.0
max_abs_error = float(np.max(np.abs(out_host.astype(np.float32) - golden.astype(np.float32))))
np.testing.assert_allclose(out_host, golden, atol=0.02, rtol=0)

gemm_result = {
    "status": "PASS",
    "actual_backend": "pyACL/acl.blas.gemm_ex device path; kernel class not inferred",
    "device_id": int(session.current_device),
    "shape": [M, N],
    "dtype": "float16",
    "repeat": REPEAT,
    "device_mean_ms": elapsed_ms / REPEAT,
    "reference_ms": reference_ms,
    "max_abs_error": max_abs_error,
    "fallback": 0,
}
print("GEMM_REPORT=" + json.dumps(gemm_result, ensure_ascii=False))


### 课后实践

学生修改：把参数单元格中的 `M、N、K` 改为 `32、64、128`，先运行参数单元格，再从“准备输入”开始向下重新运行。单算子描述随参数更新。记录：

1. A、B、输出 shape 与 M/N/K 的对应关系；
2. `actual_backend`、Device ID、平均设备时间和最大误差；
3. 为什么 Event 计时不包含 ATC、H2D 和 D2H。

观察新的输入形状、设备输出和 NumPy 校验结果。


完成本节练习后查看：[参考答案](answer/05.02_gemm_answer.md)
